Nodes = steps
Edges = rules
State = memory


What we will build (FIRST graph)

A tiny agent that:

Receives a question

Decides:

math → calculator

knowledge → lookup

Returns an answer

In [1]:
!pip install -U langgraph langchain langchain-community langchain-core


In [2]:
from typing import TypedDict, Literal

from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage

from langgraph.graph import StateGraph, END


C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
llm = ChatOllama(model="llama3", temperature=0)


C:\Users\KRISHNENDU\AppData\Local\Temp\ipykernel_19260\1566199341.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama3", temperature=0)


In [4]:
class AgentState(TypedDict):
    input: str
    output: str


In [5]:
def calculator_tool(text: str) -> str:
    try:
        return str(eval(text))
    except:
        return "Invalid math expression"


def lookup_tool(text: str) -> str:
    knowledge = {
        "langchain": "LangChain is a framework for building LLM-powered apps.",
        "ollama": "Ollama lets you run LLMs locally."
    }
    for k, v in knowledge.items():
        if k in text.lower():
            return v
    return "I don't know."


In [6]:
def router(state: AgentState) -> Literal["calculator", "lookup"]:
    text = state["input"]

    if any(c in text for c in "+-*/"):
        return "calculator"
    return "lookup"


In [7]:
def calculator_node(state: AgentState) -> AgentState:
    result = calculator_tool(state["input"])
    return {"output": result}


def lookup_node(state: AgentState) -> AgentState:
    result = lookup_tool(state["input"])
    return {"output": result}


In [11]:
graph = StateGraph(AgentState)

# Nodes
graph.add_node("router", lambda state: state)   # router node (pass-through)
graph.add_node("calculator", calculator_node)
graph.add_node("lookup", lookup_node)

# Entry point
graph.set_entry_point("router")

# Conditional routing
graph.add_conditional_edges(
    "router",
    router,   # routing function
    {
        "calculator": "calculator",
        "lookup": "lookup"
    }
)

# End states
graph.add_edge("calculator", END)
graph.add_edge("lookup", END)

app = graph.compile()



In [12]:
app.invoke({"input": "123 * 45 + 67"})


{'input': '123 * 45 + 67', 'output': '5602'}

In [13]:
app.invoke({"input": "What is LangChain?"})


{'input': 'What is LangChain?',
 'output': 'LangChain is a framework for building LLM-powered apps.'}